# Formal Spec Pipeline with LLM Judge

**Architecture:**
- **LLM 1** — Formal Spec Generator (3-step chain-of-thought: Annotation → Lifting → Translation)
- **LLM 2** — Summariser (formal specs → human-readable draft summary)
- **LLM 3 — Judge** — Independent formal methods evaluator that compares DRAFT vs GROUND TRUTH on structured verification dimensions:
  - **Precondition Accuracy (35%)**: Guards, domain membership ($u \notin \text{dom}(R)$), role checks, and logic/negation flips
  - **Postcondition Accuracy (35%)**: State transitions, map updates ($R', T', \text{Qual}', \text{Compl}'$), and lifecycle states
  - **State Variables (15%)**: Comprehensive tracking of system state variables ($C, P, M, T, \text{Qual}, \text{Compl}$)
  - **Purpose Alignment (15%)**: Intent and actor fidelity
  - **Contradiction Detection**: Explicitly flags semantic inversions or negated logic
- **Refinement Loop** — Targeted diagnostic feedback from the LLM Judge feeds back into LLM 1 until threshold is met or max iterations reached.

## 0. Install & Import

In [1]:
!pip install --upgrade groq pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.9 MB/s eta 0:00:00


In [2]:
import os, json, time, re
from typing import List, Dict, Any, Optional
from collections import defaultdict

import pandas as pd
from groq import Groq

# ── CONFIGURATION ──────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    secrets_client = UserSecretsClient()
    GROQ_API_KEY = secrets_client.get_secret("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")

if not GROQ_API_KEY:
    raise ValueError("Missing GROQ_API_KEY in Kaggle Secrets or environment.")

# Groq model fallback chain
MODEL_CHAIN          = ['llama-3.3-70b-versatile', 'llama-3.1-8b-instant']

# ── PIPELINE SETTINGS ──────────────────────────────────────────────
SIMILARITY_THRESHOLD = 0.85   # LLM Judge weighted score threshold to PASS
MAX_ITERATIONS       = 5
BATCH_SIZE           = 3      # LLM batch size — keeps under Groq rate limits
INTER_BATCH_SLEEP    = 8      # seconds between LLM batches

# ── FILE PATHS ─────────────────────────────────────────────────────
# Place your ground-truth summary.json in /kaggle/input/<your-dataset>/
# and update DATASET_FOLDER below.
DATASET_FOLDER     = "/datasets/chandrimanandi/summary-actual"   # <-- CHANGE THIS
GROUND_TRUTH_PATH  = f"/kaggle/input/{DATASET_FOLDER}/summary.json"
FORMAL_SPECS_PATH  = '/kaggle/working/formal_specifications.json'
DRAFT_SUMMARY_PATH = '/kaggle/working/draft_summary.json'
JUDGE_REPORT_PATH  = '/kaggle/working/judge_report.json'
ITER_LOG_PATH      = '/kaggle/working/iteration_log.json'
OUTPUT_DIR         = '/kaggle/working/'

# ── JUDGE LLM RUBRIC WEIGHTS ───────────────────────────────────────
# Evaluates formal specification equivalence across 4 core dimensions:
WEIGHT_PRECONDITION   = 0.35  # Guards, domain checks, boolean criteria, negations
WEIGHT_POSTCONDITION  = 0.35  # State transitions, map updates, primed variables
WEIGHT_STATE_VARS     = 0.15  # Correct system state variables identified
WEIGHT_PURPOSE        = 0.15  # Semantic operation goal and actor alignment

# ── INIT GROQ CLIENT ───────────────────────────────────────────────
client           = Groq(api_key=GROQ_API_KEY)
active_model_idx = 0

def current_model() -> str:
    return MODEL_CHAIN[active_model_idx]

print(f'Provider : Groq')
print(f'Models   : {MODEL_CHAIN}')
print(f'Threshold: {SIMILARITY_THRESHOLD}  Batch: {BATCH_SIZE}  Sleep: {INTER_BATCH_SLEEP}s')
print(f'Judge    : Multi-Dimensional LLM Judge ({current_model()})')

Provider : Groq
Models   : ['llama-3.3-70b-versatile', 'llama-3.1-8b-instant']
Threshold: 0.9  Batch: 3  Sleep: 8s
Judge    : STS(cross-encoder/stsb-roberta-large) + NLI(cross-encoder/nli-deberta-v3-base)


## 1. Helper Utilities

In [3]:
DAILY_QUOTA_EXHAUSTED = False

def _is_rpm_error(msg: str) -> bool:
    low = msg.lower()
    return '429' in msg and ('rate_limit' in low or 'rate limit' in low or 'too many requests' in low)

def _is_hard_quota_error(msg: str) -> bool:
    low = msg.lower()
    return '429' in msg and 'exceeded' in low and 'rate_limit' not in low

def _retry_wait(msg: str, default: int = 35) -> int:
    m = re.search(r'try again in ([\d.]+)s', msg)
    if m:
        return int(float(m.group(1))) + 2
    m = re.search(r'retry.after.*?(\d+)', msg.lower())
    if m:
        return int(m.group(1)) + 2
    return default


def call_llm(prompt: str,
             few_shot: Optional[List[Dict]] = None,
             max_retries: int = 3) -> str:
    """
    Groq-backed LLM caller (OpenAI-compatible chat format).
    RPM errors do NOT burn a retry attempt — just sleep and retry same model.
    Returns response text, or '' on failure.
    """
    global active_model_idx, DAILY_QUOTA_EXHAUSTED
    if DAILY_QUOTA_EXHAUSTED:
        print('   🚫 All models exhausted.')
        return ''

    def build_messages(prompt_text):
        messages = []
        if few_shot:
            for turn in few_shot:
                messages.append({
                    'role': turn['role'],
                    'content': turn['parts'][0]['text']
                })
        messages.append({'role': 'user', 'content': prompt_text})
        return messages

    attempt = 0
    while attempt < max_retries:
        attempt += 1
        model = current_model()
        try:
            response = client.chat.completions.create(
                model=model,
                messages=build_messages(prompt),
                temperature=0.2,
                max_tokens=8192,
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            msg = str(e)
            print(f'   ⚠️  [{model}] attempt {attempt}: {msg[:180]}')

            if _is_rpm_error(msg):
                wait = _retry_wait(msg)
                print(f'   ⏱️  RPM limit → sleeping {wait}s...')
                time.sleep(wait)
                attempt -= 1  # RPM waits don't count as failed attempts
            elif _is_hard_quota_error(msg):
                next_idx = active_model_idx + 1
                if next_idx < len(MODEL_CHAIN):
                    print(f'   🔄 Quota on {model} → switching to {MODEL_CHAIN[next_idx]}')
                    active_model_idx = next_idx
                    attempt = 0
                else:
                    DAILY_QUOTA_EXHAUSTED = True
                    print('   🚫 All models exhausted.')
                    return ''
            elif attempt < max_retries:
                print(f'      retrying in 5s...')
                time.sleep(5)
            else:
                print(f'   ❌ Fatal after {max_retries} attempts.')
    return ''


def parse_json(raw: str) -> Any:
    """Tolerant JSON parser with code-block extraction, comment stripping, missing-comma repair, and auto-closure."""
    raw = raw.strip()
    m = re.search(r'```(?:json)?\s*([\s\S]*?)\s*```', raw)
    if m:
        candidate = m.group(1).strip()
    else:
        m_arr = re.search(r'(\[[\s\S]*\])', raw)
        m_obj = re.search(r'(\{[\s\S]*\})', raw)
        if m_arr and (not m_obj or m_arr.start() <= m_obj.start()):
            candidate = m_arr.group(1).strip()
        elif m_obj:
            candidate = m_obj.group(1).strip()
        else:
            candidate = raw

    try:
        return json.loads(candidate)
    except Exception:
        pass

    # Fix invalid backslashes (e.g. LaTeX/formal symbols like \in, \mapsto, T \ {token})
    cleaned = re.sub(r'\\(?![\\"/bfnrt]|u[0-9a-fA-F]{4})', r'\\\\', candidate)
    # Clean single-line comments
    cleaned = re.sub(r'//.*?\n', '\n', cleaned)
    # Fix missing comma between objects: } { -> }, {
    cleaned = re.sub(r'\}\s*\{', '}, {', cleaned)
    # Fix missing comma between arrays: ] [ -> ], [
    cleaned = re.sub(r'\]\s*\[', '], [', cleaned)
    # Fix missing comma between strings: "a" "b" -> "a", "b"
    cleaned = re.sub(r'"\s*\n+\s*"', '", "', cleaned)
    cleaned = re.sub(r'"\s{1,}"', '", "', cleaned)
    # Remove trailing commas before } or ]
    cleaned = re.sub(r',\s*([\}\]])', r'\1', cleaned)

    try:
        return json.loads(cleaned)
    except Exception:
        pass

    # Try closing truncated brackets/quotes
    s = cleaned
    quotes = len(re.findall(r'(?<!\\)"', s))
    if quotes % 2 != 0:
        s += '"'

    for suffix in [']}]', ']}', '}]', ']', '}']:
        try:
            return json.loads(s.rstrip().rstrip(',') + suffix)
        except Exception:
            pass

    return json.loads(candidate)

def save_json(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f'   💾 Saved → {path}')

def load_json(path):
    assert os.path.exists(path), f"File not found: {path}"
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def batched(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i+size]

print('✅ Helpers ready.')

✅ Helpers ready.


In [4]:
print('Available Groq models:')
try:
    for m in client.models.list().data:
        print(f'  {m.id}')
except Exception as e:
    print(f'Could not list models: {e}')

print('\nSmoke test...')
test = call_llm('Reply with just the word: OK')
print(f'Response : "{test}"')
print(f'Model    : {current_model()}')

Available Groq models:
  openai/gpt-oss-120b
  moonshotai/kimi-k2-instruct-0905
  openai/gpt-oss-20b
  llama-3.1-8b-instant
  allam-2-7b
  canopylabs/orpheus-v1-english
  meta-llama/llama-4-scout-17b-16e-instruct
  groq/compound-mini
  whisper-large-v3
  openai/gpt-oss-safeguard-20b
  groq/compound
  moonshotai/kimi-k2-instruct
  meta-llama/llama-prompt-guard-2-86m
  meta-llama/llama-prompt-guard-2-22m
  qwen/qwen3-32b
  canopylabs/orpheus-arabic-saudi
  llama-3.3-70b-versatile
  whisper-large-v3-turbo

Smoke test...
Response : "OK"
Model    : llama-3.3-70b-versatile


## 2. Load Ground Truth

In [5]:
ground_truth = load_json(GROUND_TRUTH_PATH)
all_operations: List[Dict] = []
for sec in ground_truth['sections']:
    for op in sec['operations']:
        all_operations.append({
            'sectionId': sec['sectionId'],
            'sectionTitle': sec['sectionTitle'],
            **op
        })

print(f'Ground truth: {len(ground_truth["sections"])} sections, {len(all_operations)} operations')
for s in ground_truth['sections']:
    print(f'  [{s["sectionId"]}] {s["sectionTitle"]} — {len(s["operations"])} ops')

Ground truth: 4 sections, 21 operations
  [1] Authentication — 3 ops
  [2] Plumber Onboarding Wizard (Steps 1–6) — 1 ops
  [3] Plumber Qualification — 7 ops
  [4] Complaint Lifecycle — 10 ops


## 3. LLM 1 — Formal Spec Generator

3-step chain-of-thought:
- **Step 1 — Annotation**: Extract actionable requirement sentences.
- **Step 2 — Lifting**: Convert sentences to Lifted-NL temporal/logical statements.
- **Step 3 — Translation**: Translate Lifted-NL to formal specs (Precondition, API call, Postcondition).

In [6]:
# ── STEP 1: ANNOTATION ────────────────────────────────────────────

ANNOT_SCHEMA = json.dumps({
    'type': 'array',
    'items': {
        'type': 'object',
        'properties': {
            'operationId': {'type': 'string'},
            'operationName': {'type': 'string'},
            'actionable_sentences': {
                'type': 'array', 'items': {'type': 'string'},
                'description': 'Sentences encoding state transitions, guards, or system actions.'
            }
        },
        'required': ['operationId', 'operationName', 'actionable_sentences']
    }
}, indent=2)

FS_ANNOT = [
    {'role': 'user', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'name': 'Signup',
        'purpose': 'Register a new account.',
        'precondition': 'The username u must not already exist in R.',
        'postcondition': 'R now contains an entry mapping u to hashed password p.'
    }])}]},
    {'role': 'assistant', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'operationName': 'Signup',
        'actionable_sentences': [
            'The username u must not already exist in R.',
            'R now contains an entry mapping u to hashed password p.'
        ]
    }])}]}
]

def step1_annotate(operations: List[Dict], extra: str = '') -> List[Dict]:
    print('\n🔹 STEP 1: Annotation')
    sys_p = f"""You are an expert in software engineering and formal methods.
Extract ONLY sentences that are actionable system requirements:
  • State transitions  ("state becomes X", "advances to Y", "reverts to Z")
  • Guards/Preconditions ("must be in state X", "must not already exist", "must match")
  • Postconditions ("is removed", "is created", "field is set to", "added to map")
Ignore introductory or descriptive sentences.
{extra}
Output ONLY a raw JSON array of objects (no markdown, no schema, no preamble) like:
[
  {{
    "operationId": "1-A",
    "operationName": "Signup",
    "actionable_sentences": [
      "The username u must not already exist in R.",
      "R now contains an entry mapping u to hashed password p."
    ]
  }}
]"""

    results = []
    bs = list(batched(operations, BATCH_SIZE))
    for i, batch in enumerate(bs):
        if DAILY_QUOTA_EXHAUSTED: break
        print(f'   Batch {i+1}/{len(bs)}: {len(batch)} ops')
        inp = [{'operationId': op['operationId'], 'name': op['name'],
                'purpose': op['purpose'], 'precondition': op['precondition'],
                'postcondition': op['postcondition']} for op in batch]

        batch_parsed = False
        cleaned = []
        for attempt in range(2):
            raw = call_llm(sys_p + '\nJSON input:\n' + json.dumps(inp, indent=2), few_shot=FS_ANNOT)
            if not raw: continue
            try:
                r = parse_json(raw)
                if isinstance(r, dict):
                    for val in r.values():
                        if isinstance(val, list) and val and isinstance(val[0], dict):
                            r = val
                            break
                    else:
                        r = [r]
                elif not isinstance(r, list):
                    r = [r]

                cleaned = []
                for idx, item in enumerate(r):
                    if not isinstance(item, dict): continue
                    op_id = item.get('operationId') or item.get('operation_id') or item.get('id')
                    op_name = item.get('operationName') or item.get('name')
                    if not op_id and idx < len(batch):
                        op_id = batch[idx]['operationId']
                        op_name = op_name or batch[idx]['name']
                    if op_id:
                        item['operationId'] = op_id
                        if op_name: item['operationName'] = op_name
                        cleaned.append(item)

                if cleaned:
                    batch_parsed = True
                    break
            except Exception as e:
                print(f'   ❌ Parse attempt {attempt+1}: {e} | raw[:150]: {raw[:150]}')
                time.sleep(2)

        # Fallback: if parsing failed, extract precondition & postcondition directly
        if not batch_parsed or not cleaned:
            print(f'   ⚠️ Fallback activated for Batch {i+1}: extracting sentences directly from input')
            cleaned = []
            for op in batch:
                sentences = [s for s in [op.get('precondition'), op.get('postcondition')] if s]
                cleaned.append({
                    'operationId': op['operationId'],
                    'operationName': op['name'],
                    'actionable_sentences': sentences or [op.get('purpose', '')]
                })

        results.extend(cleaned)
        print(f'   ✅ {sum(len(x.get("actionable_sentences",[])) for x in cleaned)} sentences')
        if i < len(bs)-1: time.sleep(INTER_BATCH_SLEEP)

    print(f'   Total annotated ops: {len(results)}')
    return results

print('✅ Step 1 defined.')

✅ Step 1 defined.


In [7]:
# ── STEP 2: LIFTING ───────────────────────────────────────────────

LIFT_SYS = """You are an expert in formal methods and temporal logic.
Convert each actionable sentence to a Lifted Natural Language (Lifted NL) statement.
Rules:
  • Modal vocabulary: 'always','whenever','implies','must hold that','iff'
  • State changes: 'X becomes Y', 'X is added to map M', 'X is removed from M'
  • Named state vars: T (session token map), Qual (qualification map), Compl (complaint map),
    C/P/M (credential maps), QualState, ComplaintRecord
  • Actors: customer c, plumber p, manager m
  • One logical sentence per item.
Output ONLY a raw JSON array of objects (no markdown, no schema, no preamble) like:
[
  {
    "operationId": "1-A",
    "operationName": "Signup",
    "source_sentence": "The username u must not already exist in R.",
    "lifted_nl": "whenever signup(u,p,role) is called, it must hold that u ∉ dom(R) where R=roleMap(role)"
  }
]"""

FS_LIFT = [
    {'role': 'user', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'operationName': 'Signup',
        'source_sentence': 'The username u must not already exist in R.'
    }, {
        'operationId': '1-A', 'operationName': 'Signup',
        'source_sentence': 'R now contains an entry mapping u to hashed password p.'
    }])}]},
    {'role': 'assistant', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'operationName': 'Signup',
        'source_sentence': 'The username u must not already exist in R.',
        'lifted_nl': 'whenever signup(u,p,role) is called, it must hold that u ∉ dom(R) where R=roleMap(role)'
    }, {
        'operationId': '1-A', 'operationName': 'Signup',
        'source_sentence': 'R now contains an entry mapping u to hashed password p.',
        'lifted_nl': 'always after signup(u,p,role) succeeds, R becomes R ∪ {u ↦ hash(p)}'
    }])}]}
]

def step2_lift(annotated: List[Dict]) -> List[Dict]:
    print('\n🔹 STEP 2: Lifting')
    # Cap at 4 actionable sentences per operation to prevent token explosion during refinement
    flat = [{'operationId': x['operationId'], 'operationName': x.get('operationName', x.get('name', '')),
             'source_sentence': s}
            for x in annotated for s in x.get('actionable_sentences', [])[:4]
            if isinstance(x, dict) and ('operationId' in x or 'operation_id' in x)]
    print(f'   Sentences to lift: {len(flat)}')
    results = []
    bs = list(batched(flat, BATCH_SIZE))
    for i, batch in enumerate(bs):
        if DAILY_QUOTA_EXHAUSTED: break
        print(f'   Batch {i+1}/{len(bs)}: {len(batch)} sentences')

        batch_parsed = False
        cleaned = []
        for attempt in range(2):
            raw = call_llm(LIFT_SYS + '\nSentences:\n' + json.dumps(batch, indent=2), few_shot=FS_LIFT)
            if not raw: continue
            try:
                r = parse_json(raw)
                if isinstance(r, dict):
                    for val in r.values():
                        if isinstance(val, list) and val and isinstance(val[0], dict):
                            r = val
                            break
                    else:
                        r = [r]
                elif not isinstance(r, list):
                    r = [r]

                cleaned = []
                for idx, item in enumerate(r):
                    if not isinstance(item, dict): continue
                    op_id = item.get('operationId') or item.get('operation_id') or item.get('op_id') or item.get('id')
                    op_name = item.get('operationName') or item.get('operation_name') or item.get('name')
                    if not op_id:
                        src = item.get('source_sentence')
                        match = next((b for b in batch if b.get('source_sentence') == src), None) if src else None
                        if match:
                            op_id = match['operationId']
                            op_name = op_name or match.get('operationName', '')
                        elif idx < len(batch):
                            op_id = batch[idx]['operationId']
                            op_name = op_name or batch[idx].get('operationName', '')
                        elif batch:
                            op_id = batch[0]['operationId']
                            op_name = op_name or batch[0].get('operationName', '')
                    if op_id:
                        item['operationId'] = op_id
                        if op_name: item['operationName'] = op_name
                        cleaned.append(item)

                if cleaned:
                    batch_parsed = True
                    break
            except Exception as e:
                print(f'   ❌ Parse attempt {attempt+1}: {e} | raw[:150]: {raw[:150]}')
                time.sleep(2)

        # Fallback: if lifting failed, use source sentences directly
        if not batch_parsed or not cleaned:
            print(f'   ⚠️ Fallback activated for Batch {i+1}: passing sentences through directly')
            cleaned = []
            for item in batch:
                cleaned.append({
                    'operationId': item['operationId'],
                    'operationName': item.get('operationName', ''),
                    'source_sentence': item['source_sentence'],
                    'lifted_nl': item['source_sentence']
                })

        results.extend(cleaned)
        print(f'   ✅ {len(cleaned)} lifted')
        if i < len(bs)-1: time.sleep(INTER_BATCH_SLEEP)

    print(f'   Total lifted: {len(results)}')
    return results

print('✅ Step 2 defined.')

✅ Step 2 defined.


In [8]:
# ── STEP 3: TRANSLATION ───────────────────────────────────────────

TRANS_SYS = """You are an expert Formal Specification Translator (predicate logic + set theory).

GLOBAL STATE:
  C,P,M : Map<username,hashedPassword>  (Customer/Plumber/Manager credentials)
  T     : Map<token,username>           (active session tokens)
  Qual  : Map<plumberId,QualState>
    QualState in {STEP_1..STEP_6, ONBOARD_DONE, Q_INIT, UNDER_EXAM, TRAINING, ACTIVE, SUSPENDED, REJECTED}
  Compl : Map<complaintId,ComplaintRecord>
    ComplaintRecord.state in {RAISED,ASSIGNED,UNDER_EXAM_C,QUOTATION_RAISED,UNDER_EXEC,PAYMENT_PENDING,DONE,ABANDONED}

RULES:
  1. LABEL        : ALL_CAPS_SNAKE_CASE unique per operation (e.g. LOGIN_OK)
  2. Precondition : pure Boolean predicate, no prose. Use set/predicate logic notation.
  3. Function     : name(param1:Type1, ...) -> ReturnType [HTTP_CODE]
  4. Postcondition: use primed vars, list ALL four maps even if unchanged
     e.g. T'=T union {tok|->u} AND C'=C AND Qual'=Qual AND Compl'=Compl
  5. Merge ALL lifted_nl items for the SAME operationId into ONE output record.

Output ONLY a raw JSON array of objects (no markdown, no schema, no preamble) like:
[
  {
    "operationId": "1-A",
    "operationName": "Signup",
    "LABEL": "SIGNUP_OK",
    "lifted_nl": "whenever signup(u,p,role) called, u not in dom(R); after success R becomes R union {u|->hash(p)}",
    "Precondition": "role in {Customer,Plumber,Manager} AND R=roleMap(role) AND u not in dom(R)",
    "Function": "signup(u:Username, p:Password, role:Role) -> Unit [201 Created]",
    "Postcondition": "R'=R union {u|->hash(p)} AND T'=T AND Qual'=Qual AND Compl'=Compl"
  }
]"""

FS_TRANS = [
    {'role': 'user', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'operationName': 'Signup',
        'lifted_nl': 'whenever signup(u,p,role) called, u not in dom(R); after success R becomes R union {u|->hash(p)}'
    }])}]},
    {'role': 'assistant', 'parts': [{'text': json.dumps([{
        'operationId': '1-A', 'operationName': 'Signup',
        'LABEL': 'SIGNUP_OK',
        'lifted_nl': 'whenever signup(u,p,role) called, u not in dom(R); after success R becomes R union {u|->hash(p)}',
        'Precondition': 'role in {Customer,Plumber,Manager} AND R=roleMap(role) AND u not in dom(R)',
        'Function': 'signup(u:Username, p:Password, role:Role) -> Unit [201 Created]',
        'Postcondition': "R'=R union {u|->hash(p)} AND T'=T AND Qual'=Qual AND Compl'=Compl"
    }])}]}
]

def step3_translate(lifted: List[Dict]) -> List[Dict]:
    print('\n🔹 STEP 3: Translation -> Formal Specs')
    grouped = defaultdict(list)
    for x in lifted:
        if not isinstance(x, dict): continue
        op_id = x.get('operationId') or x.get('operation_id') or x.get('op_id') or x.get('id')
        if op_id:
            x['operationId'] = op_id
            grouped[op_id].append(x)
        else:
            print(f'   ⚠️ Warning: skipped lifted item without operationId: {x}')
    op_groups = list(grouped.values())
    print(f'   Operations to formalise: {len(op_groups)}')
    results = []
    bs = list(batched(op_groups, BATCH_SIZE))
    for i, batch_ops in enumerate(bs):
        if DAILY_QUOTA_EXHAUSTED: break
        items = [x for ops in batch_ops for x in ops]
        print(f'   Batch {i+1}/{len(bs)}: {len(batch_ops)} ops ({len(items)} items)')

        batch_parsed = False
        cleaned = []
        for attempt in range(2):
            raw = call_llm(TRANS_SYS + '\nLifted-NL:\n' + json.dumps(items, indent=2), few_shot=FS_TRANS)
            if not raw: continue
            try:
                r = parse_json(raw)
                if isinstance(r, dict):
                    for val in r.values():
                        if isinstance(val, list) and val and isinstance(val[0], dict):
                            r = val
                            break
                    else:
                        r = [r]
                elif not isinstance(r, list):
                    r = [r]
                cleaned = []
                for idx, spec in enumerate(r):
                    if not isinstance(spec, dict): continue
                    op_id = spec.get('operationId') or spec.get('operation_id') or spec.get('op_id') or spec.get('id')
                    if not op_id and idx < len(batch_ops):
                        op_id = batch_ops[idx][0].get('operationId')
                    if op_id:
                        spec['operationId'] = op_id
                        cleaned.append(spec)
                if cleaned:
                    batch_parsed = True
                    break
            except Exception as e:
                print(f'   ❌ Parse attempt {attempt+1}: {e} | raw[:150]: {raw[:150]}')
                time.sleep(2)

        # Fallback: synthesize baseline formal spec so operation is never lost
        if not batch_parsed or not cleaned:
            print(f'   ⚠️ Fallback activated for Batch {i+1}: synthesizing formal spec skeletons')
            cleaned = []
            for op_items in batch_ops:
                op_id = op_items[0].get('operationId', 'UNKNOWN')
                op_name = op_items[0].get('operationName', op_id)
                lifted_text = '; '.join(it.get('lifted_nl', '') for it in op_items)
                clean_name = re.sub(r'[^A-Z0-9]+', '_', op_name.upper()).strip('_')
                cleaned.append({
                    'operationId': op_id,
                    'operationName': op_name,
                    'LABEL': f'{clean_name}_OK',
                    'lifted_nl': lifted_text,
                    'Precondition': 'true',
                    'Function': f"{re.sub(r'[^a-zA-Z0-9]+', '', op_name)}() -> Unit [200 OK]",
                    'Postcondition': "C'=C AND P'=P AND M'=M AND T'=T AND Qual'=Qual AND Compl'=Compl"
                })

        results.extend(cleaned)
        print(f'   ✅ {len(cleaned)} specs')
        if i < len(bs)-1: time.sleep(INTER_BATCH_SLEEP)

    print(f'   Total formal specs: {len(results)}')
    return results

print('✅ Step 3 defined.')

✅ Step 3 defined.


In [9]:
print('='*60)
print('LLM 1 — FORMAL SPEC GENERATOR (initial run)')
print('='*60)

annotated_ops = step1_annotate(all_operations)
lifted_items  = step2_lift(annotated_ops)
formal_specs  = step3_translate(lifted_items)

save_json(formal_specs, FORMAL_SPECS_PATH)

if DAILY_QUOTA_EXHAUSTED:
    print('\n🚫 Halted — quota exhausted on all models.')
else:
    print(f'\n✅ LLM 1 done. {len(formal_specs)} specs -> {FORMAL_SPECS_PATH}')

LLM 1 — FORMAL SPEC GENERATOR (initial run)

🔹 STEP 1: Annotation
   Batch 1/7: 3 ops
   ✅ 6 sentences
   Batch 2/7: 3 ops
   ✅ 7 sentences
   Batch 3/7: 3 ops
   ✅ 6 sentences
   Batch 4/7: 3 ops
   ✅ 6 sentences
   Batch 5/7: 3 ops
   ✅ 7 sentences
   Batch 6/7: 3 ops
   ✅ 6 sentences
   Batch 7/7: 3 ops
   ✅ 6 sentences
   Total annotated ops: 21

🔹 STEP 2: Lifting
   Sentences to lift: 44
   Batch 1/15: 3 sentences
   ✅ 3 lifted
   Batch 2/15: 3 sentences
   ✅ 3 lifted
   Batch 3/15: 3 sentences
   ✅ 3 lifted
   Batch 4/15: 3 sentences
   ✅ 3 lifted
   Batch 5/15: 3 sentences
   ✅ 3 lifted
   Batch 6/15: 3 sentences
   ✅ 3 lifted
   Batch 7/15: 3 sentences
   ✅ 3 lifted
   Batch 8/15: 3 sentences
   ❌ Parse: Invalid \escape: line 12 column 84 (char 482) | raw[:150]: [
  {
    "operationId": "3-C3",
    "operationName": "Delete Plumber Account",
    "source_sentence": "The plumber must exist in the qualification ma
   Batch 9/15: 3 sentences
   ✅ 3 lifted
   Batch 10/15: 3 sentences

## 4. LLM 2 — Summariser

Converts formal specs back into a `draft_summary.json` that mirrors the schema of `summary.json`.

In [10]:
SUM_PROMPT = """You are an expert technical writer.
Given formal specs, reconstruct a human-readable summary JSON in EXACTLY this schema:
{
  "sections": [
    {
      "sectionId": <int>,
      "sectionTitle": <string>,
      "operations": [
        {
          "operationId": <string — MUST match input>,
          "name": <string>,
          "purpose": <string — 1-2 plain-English sentences>,
          "stateVariables": [<string>, ...],
          "precondition": <string — plain English, no math>,
          "postcondition": <string — plain English, no math>
        }
      ]
    }
  ]
}
Section grouping:
  1 — Authentication        (1-A, 1-B, 1-C)
  2 — Plumber Onboarding    (2-k)
  3 — Plumber Qualification (3-A..3-C3)
  4 — Complaint Lifecycle   (4-A..4-F)
Include EVERY operationId. Output ONLY raw JSON — no markdown, no preamble.
"""

def run_llm2(specs: List[Dict]) -> Dict:
    print('\n' + '='*60 + '\nLLM 2 — SUMMARISER\n' + '='*60)
    if DAILY_QUOTA_EXHAUSTED: return {'sections': []}
    raw = call_llm(SUM_PROMPT + '\nSpecs:\n' + json.dumps(specs, indent=2))
    if not raw: return {'sections': []}
    try:
        d = parse_json(raw)
        n = sum(len(s.get('operations', [])) for s in d.get('sections', []))
        print(f'   ✅ {len(d.get("sections",[]))} sections, {n} ops')
        return d
    except Exception as e:
        print(f'   ❌ Parse: {e}')
        return {'sections': []}

print('✅ LLM 2 defined.')

✅ LLM 2 defined.


## 5. LLM 3 — Judge (Multi-Dimensional Semantic & Formal Verification Evaluator)

Evaluates candidate operations in **DRAFT** against **GROUND TRUTH** using a formal methods rubric:
- **Precondition Accuracy (35%)**: Validates guards, domain membership ($u \notin \text{dom}(R)$), role checks, and catches logic/negation flips.
- **Postcondition Accuracy (35%)**: Evaluates state transitions, map updates ($R', T', \text{Qual}', \text{Compl}'$), and lifecycle states.
- **State Variables (15%)**: Ensures correct variables ($C, P, M, T, \text{Qual}, \text{Compl}$) are tracked.
- **Purpose Alignment (15%)**: Ensures operational intent and actor roles match ground truth.
- **Semantic Contradiction Detection**: Explicitly catches and penalizes inverted requirements.
- **Actionable Diagnostics**: Produces concrete, per-operation feedback for the refinement loop.

In [11]:
# ── JUDGE LLM PROMPT & SYSTEM INSTRUCTIONS ────────────────────────

JUDGE_SYSTEM_PROMPT = """You are an expert Formal Verification & Formal Methods Judge.
Your task is to critically evaluate candidate DRAFT specification operations against canonical GROUND TRUTH specifications.

SYSTEM DOMAIN CONTEXT:
The system models a verified service platform with the following state maps:
  • C, P, M : Map<username, hashedPassword> (Customer, Plumber, Manager credentials)
  • T       : Map<token, username> (active authentication session tokens)
  • Qual    : Map<plumberId, QualState>
    Valid QualStates: {STEP_1..STEP_6, ONBOARD_DONE, Q_INIT, UNDER_EXAM, TRAINING, ACTIVE, SUSPENDED, REJECTED}
  • Compl   : Map<complaintId, ComplaintRecord>
    Valid Complaint states: {RAISED, ASSIGNED, UNDER_EXAM_C, QUOTATION_RAISED, UNDER_EXEC, PAYMENT_PENDING, DONE, ABANDONED}

EVALUATION RUBRIC (Score each dimension strictly between 0.0 and 1.0):
1. Precondition Accuracy (Weight: 0.35):
   • Do precondition guards, domain checks (e.g., u ∉ dom(R) vs u ∈ dom(R)), session token requirements, and state prerequisites match ground truth?
   • CRITICAL: Flag any negation or inverted logic (e.g., checking that an entity exists when it must NOT exist).
2. Postcondition Accuracy (Weight: 0.35):
   • Do state transitions match the ground truth? Are updated maps (C', P', M', T', Qual', Compl') specified correctly?
   • Are target states exact (e.g., TRAINING vs ACTIVE, ASSIGNED vs UNDER_EXEC)?
3. State Variables Accuracy (Weight: 0.15):
   • Are all involved state variables (C, P, M, T, Qual, Compl, role maps) identified and correctly referenced?
4. Purpose & Intent Alignment (Weight: 0.15):
   • Does the candidate purpose faithfully represent the operational intent and actor role?

CONTRADICTION / INVERSION DETECTION:
Set `contradiction_detected: true` if the draft asserts the logical inverse of ground truth (e.g. inverted precondition guard, wrong transition direction, negated state requirement). If true, provide a clear `contradiction_explanation`.

OUTPUT FORMAT:
Output ONLY a raw JSON array of evaluation objects (one per operation in the input batch) matching EXACTLY this schema:
[
  {
    "operationId": "<string>",
    "name": "<string>",
    "dimension_scores": {
      "precondition_accuracy": <float 0.0-1.0>,
      "postcondition_accuracy": <float 0.0-1.0>,
      "state_variables_accuracy": <float 0.0-1.0>,
      "purpose_accuracy": <float 0.0-1.0>
    },
    "contradiction_detected": <true or false>,
    "contradiction_explanation": "<string explaining the exact logical inversion if detected, else empty string>",
    "issues": ["<concise bullet of discrepancy 1>", "<concise bullet of discrepancy 2>"],
    "feedback": "<prescriptive 1-2 sentence guidance for LLM 1 on how to correct the formal specification>"
  }
]
No markdown wrapping, no explanation outside the JSON array.
"""

FS_JUDGE = [
    {
        'role': 'user',
        'parts': [{
            'text': json.dumps({
                'ground_truth': [{
                    'operationId': '1-A',
                    'name': 'Signup',
                    'purpose': 'Register a new account. The role parameter determines which credential map (C, P, or M) is updated.',
                    'stateVariables': ['R ∈ {C, P, M} depending on role'],
                    'precondition': 'The username u must not already exist in the target role map R.',
                    'postcondition': 'The credential map R now contains an entry mapping u to hashed password p.'
                }],
                'draft': [{
                    'operationId': '1-A',
                    'name': 'Signup',
                    'purpose': 'Create a new user account with hashed password.',
                    'stateVariables': ['R', 'u', 'p'],
                    'precondition': 'The username must not already exist in the system.',
                    'postcondition': 'The user is added to the system with a hashed password.'
                }]
            }, indent=2)
        }]
    },
    {
        'role': 'assistant',
        'parts': [{
            'text': json.dumps([{
                'operationId': '1-A',
                'name': 'Signup',
                'dimension_scores': {
                    'precondition_accuracy': 0.70,
                    'postcondition_accuracy': 0.65,
                    'state_variables_accuracy': 0.75,
                    'purpose_accuracy': 0.85
                },
                'contradiction_detected': False,
                'contradiction_explanation': '',
                "issues": [
                    "Precondition mentions generic system instead of target role credential map R (u ∉ dom(R)).",
                    "Postcondition fails to specify that map R is updated with mapping u -> hash(p)."
                ],
                "feedback": "Specify target role map R ∈ {C, P, M} in precondition (u ∉ dom(R)) and explicitly state postcondition update R' = R ∪ {u ↦ hash(p)}."
            }], indent=2)
        }]
    }
]

print('✅ LLM 3 Judge prompt and few-shot examples defined.')

Loading STS model : cross-encoder/stsb-roberta-large


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cross-encoder/stsb-roberta-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

BERTScore judge models ready.


In [12]:
def extract_operations_dict(data: Dict) -> Dict[str, Dict]:
    """Extract {operationId: op_dict} from summary JSON."""
    ops = {}
    for section in data.get('sections', []):
        for op in section.get('operations', []):
            ops[op['operationId']] = op
    return ops


def run_llm_judge(ground_truth: Dict, draft: Dict) -> Dict:
    """
    Evaluates draft vs ground truth using the Multi-Dimensional LLM Judge.
    Processes operations in small batches to maintain precision, prevent context truncation,
    and avoid Groq rate limits.
    """
    print('\n' + '='*60 + '\nLLM 3 — JUDGE EVALUATION\n' + '='*60)

    gt_ops    = extract_operations_dict(ground_truth)
    draft_ops = extract_operations_dict(draft)

    common_ids    = sorted(set(gt_ops.keys()) & set(draft_ops.keys()))
    only_in_gt    = sorted(set(gt_ops.keys()) - set(draft_ops.keys()))
    only_in_draft = sorted(set(draft_ops.keys()) - set(gt_ops.keys()))

    print(f'   Ground truth ops : {len(gt_ops)}')
    print(f'   Draft ops        : {len(draft_ops)}')
    print(f'   Common ops       : {len(common_ids)}')
    if only_in_gt:    print(f'   Missing from draft: {only_in_gt}')
    if only_in_draft: print(f'   Extra in draft    : {only_in_draft}')

    if not common_ids:
        return {
            'overall_score': 0.0,
            'operation_scores': [],
            'missing_operations': only_in_gt,
            'overall_feedback': 'No common operations found between draft and ground truth.',
            'judge_type': 'llm_judge',
            'threshold': SIMILARITY_THRESHOLD,
        }

    operation_scores = []
    batches = list(batched(common_ids, BATCH_SIZE))
    total_batches = len(batches)

    print(f'   Evaluating {len(common_ids)} operations in {total_batches} batches...')

    for b_idx, batch_ids in enumerate(batches, 1):
        if DAILY_QUOTA_EXHAUSTED:
            print('   🚫 Quota exhausted during judge evaluation.')
            break

        print(f'   Batch {b_idx}/{total_batches}: {batch_ids}')

        batch_gt    = [gt_ops[oid] for oid in batch_ids]
        batch_draft = [draft_ops[oid] for oid in batch_ids]

        prompt_input = {
            'ground_truth': batch_gt,
            'draft': batch_draft
        }
        user_prompt = (
            JUDGE_SYSTEM_PROMPT
            + '\n\n--- BATCH TO EVALUATE ---\n'
            + json.dumps(prompt_input, indent=2)
        )

        raw = call_llm(user_prompt, few_shot=FS_JUDGE)
        batch_evals = []
        if raw:
            try:
                parsed = parse_json(raw)
                if isinstance(parsed, list):
                    batch_evals = parsed
                elif isinstance(parsed, dict) and 'operation_scores' in parsed:
                    batch_evals = parsed['operation_scores']
                elif isinstance(parsed, dict):
                    batch_evals = [parsed]
            except Exception as e:
                print(f'      ⚠️ Batch {b_idx} JSON parse error: {e}')

        # Index evaluations by operationId for robust mapping
        eval_map = {item.get('operationId'): item for item in batch_evals if isinstance(item, dict)}

        for oid in batch_ids:
            gt_item    = gt_ops[oid]
            draft_item = draft_ops.get(oid, {})
            ev         = eval_map.get(oid, {})

            dims = ev.get('dimension_scores', {})
            pre   = float(dims.get('precondition_accuracy', 0.5))
            post  = float(dims.get('postcondition_accuracy', 0.5))
            state = float(dims.get('state_variables_accuracy', 0.5))
            purp  = float(dims.get('purpose_accuracy', 0.5))

            weighted_score = round(
                WEIGHT_PRECONDITION * pre +
                WEIGHT_POSTCONDITION * post +
                WEIGHT_STATE_VARS * state +
                WEIGHT_PURPOSE * purp,
                4
            )

            contradiction = bool(ev.get('contradiction_detected', False))
            contra_expl   = ev.get('contradiction_explanation', '')
            issues        = ev.get('issues', [])
            feedback      = ev.get('feedback', '')

            if contradiction:
                final_score = min(weighted_score, 0.40)
                status = 'FAIL'
            else:
                final_score = weighted_score
                status = 'PASS' if final_score >= SIMILARITY_THRESHOLD else 'FAIL'

            if not feedback:
                if status == 'PASS':
                    feedback = 'Precondition and postcondition match ground truth specifications.'
                else:
                    feedback = f'Score {final_score:.2f} below threshold ({SIMILARITY_THRESHOLD}). Align guards and state transitions.'

            gt_summary_text = f"{gt_item.get('name','')} | Pre: {gt_item.get('precondition','')} | Post: {gt_item.get('postcondition','')}"
            draft_summary_text = f"{draft_item.get('name','')} | Pre: {draft_item.get('precondition','')} | Post: {draft_item.get('postcondition','')}"

            operation_scores.append({
                'operationId':                oid,
                'name':                       gt_item.get('name', ''),
                'score':                      final_score,
                'status':                     status,
                'dimension_scores':           dims,
                'precondition_score':         pre,
                'postcondition_score':        post,
                'state_vars_score':           state,
                'purpose_score':              purp,
                'contradiction_detected':     contradiction,
                'contradiction_explanation':  contra_expl,
                'issues':                     issues,
                'feedback':                   feedback,
                'ground_truth_text':          gt_summary_text,
                'draft_text':                 draft_summary_text,
                # Compatibility aliases for existing downstream code
                'sts_score':                  final_score,
                'nli_score':                  0.0 if contradiction else final_score,
                'nli_label':                  'contradiction' if contradiction else 'entailment',
            })

        if b_idx < total_batches:
            time.sleep(INTER_BATCH_SLEEP)

    # ── Summary statistics ──────────────────────────────────────────
    passed         = [op for op in operation_scores if op['status'] == 'PASS']
    failed         = [op for op in operation_scores if op['status'] == 'FAIL']
    contradictions = [op for op in operation_scores if op.get('contradiction_detected')]
    overall_score  = round(sum(op['score'] for op in operation_scores) / len(operation_scores), 4) if operation_scores else 0.0

    fb_parts = []
    if contradictions:
        fb_parts.append(f'{len(contradictions)} contradiction(s) detected: {[op["operationId"] for op in contradictions]}')
    if failed:
        fb_parts.append(f'{len(failed)} op(s) below threshold ({SIMILARITY_THRESHOLD}): {[op["operationId"] for op in failed]}')
    overall_feedback = '. '.join(fb_parts) if fb_parts else f'All operations passed LLM Judge threshold ({SIMILARITY_THRESHOLD}).'

    # ── Console display table ───────────────────────────────────────
    print(f'\n   {"="*70}')
    print(f'   LLM JUDGE RESULTS  |  threshold = {SIMILARITY_THRESHOLD}')
    print(f'   {"="*70}')

    df = pd.DataFrame(operation_scores)
    display_cols = ['operationId', 'status', 'score', 'precondition_score', 'postcondition_score', 'contradiction_detected']
    print(df[display_cols].to_string(index=False))

    print(f'\n   PASSED: {len(passed)}/{len(operation_scores)}')
    print(f'   FAILED: {len(failed)}/{len(operation_scores)}')
    if failed:
        print(f'   ⚠️  Failed op IDs: {[op["operationId"] for op in failed]}')
    if contradictions:
        print(f'\n   🔴 CONTRADICTIONS detected in {len(contradictions)} op(s):')
        for op in contradictions:
            print(f'      {op["operationId"]}: {op.get("contradiction_explanation")}')

    if only_in_gt:
        print(f'\n   ⚠️  Missing ops (in GT but not draft): {only_in_gt}')

    print(f'\n   📊 Overall score: {overall_score:.4f}')

    return {
        'overall_score':       overall_score,
        'operation_scores':    operation_scores,
        'missing_operations':  only_in_gt,
        'overall_feedback':    overall_feedback,
        'judge_type':          'llm_judge',
        'threshold':           SIMILARITY_THRESHOLD,
    }

# Backward-compatible aliases
run_bertscore_judge = run_llm_judge
run_llm3 = run_llm_judge

print('✅ LLM 3 Judge defined.')

✅ BERTScore judge defined.


## 6. Refinement Helper

Converts LLM Judge diagnostic feedback into targeted prompt guidance for LLM 1.

In [13]:
def build_refinement(report: Dict) -> str:
    """
    Converts LLM Judge report into targeted refinement feedback
    prepended to the LLM 1 annotation prompt.
    """
    missing  = report.get('missing_operations', [])
    feedback = report.get('overall_feedback', '')

    # Low-scoring ops or explicit fails
    low_ops  = [
        op for op in report.get('operation_scores', [])
        if op.get('status') == 'FAIL' or op.get('score', 1.0) < SIMILARITY_THRESHOLD
    ]

    # Explicit contradictions
    contradiction_ops = [
        op for op in report.get('operation_scores', [])
        if op.get('contradiction_detected') or op.get('nli_label') == 'contradiction'
    ]

    all_priority_ids = sorted(list(set([op['operationId'] for op in low_ops + contradiction_ops])))
    op_map = {op['operationId']: op for op in report.get('operation_scores', [])}

    op_feedback_lines = []
    for oid in all_priority_ids:
        op = op_map.get(oid, {})
        score = op.get('score', 0.0)
        pre   = op.get('precondition_score', 0.0)
        post  = op.get('postcondition_score', 0.0)
        issues = op.get('issues', [])
        action = op.get('feedback', '')
        contra = op.get('contradiction_explanation', '')

        line = f"  [{oid}] Score={score:.2f} (Precondition={pre:.2f}, Postcondition={post:.2f})"
        if contra:
            line += f"\n    🔴 CONTRADICTION: {contra}"
        if issues:
            line += f"\n    Discrepancies : {'; '.join(issues)}"
        if action:
            line += f"\n    Fix Direction : {action}"
        op_feedback_lines.append(line)

    focus = sorted(list(set(all_priority_ids + missing)))

    return f"""
=== LLM JUDGE REFINEMENT FEEDBACK (from previous iteration) ===
Overall Assessment: {feedback}

Operations Needing Improvement (FAIL or CONTRADICTION):
{chr(10).join(op_feedback_lines) if op_feedback_lines else '  (none specifically flagged)'}

Missing operations (present in ground truth but absent from draft): {missing or '(none)'}
Focus extra attention on operationIds: {focus}

Formal Specification Rules:
  • CONTRADICTION: Inverted pre/postcondition logic — verify negation guards and state transition directions.
  • Exact QualState names: STEP_1..STEP_6, ONBOARD_DONE, Q_INIT, UNDER_EXAM, TRAINING, ACTIVE, SUSPENDED, REJECTED
  • Exact Compl.state:     RAISED, ASSIGNED, UNDER_EXAM_C, QUOTATION_RAISED, UNDER_EXEC, PAYMENT_PENDING, DONE, ABANDONED
  • Preconditions: Specify exact role credential map domain guards (e.g. u ∉ dom(R) vs u ∈ dom(R)).
  • Postconditions: MUST list ALL four maps (C'/P'/M', T', Qual', Compl') even if unchanged.
=== END REFINEMENT FEEDBACK ===
"""

print('✅ Refinement helper defined.')

✅ Refinement helper defined.


## 7. Main Iterative Loop

Runs: **LLM 2 → LLM Judge → (if FAIL) LLM 1 refinement → repeat**

Stops when `overall_score >= SIMILARITY_THRESHOLD` or `MAX_ITERATIONS` is reached.

In [ ]:
print('\n' + '='*60)
print(f'  ITERATIVE PIPELINE  threshold={SIMILARITY_THRESHOLD}  max_iter={MAX_ITERATIONS}')
print('='*60)

iter_log      = []
current_score = 0.0
current_specs = formal_specs

for iteration in range(1, MAX_ITERATIONS + 1):
    print(f'\n--- ITERATION {iteration}/{MAX_ITERATIONS} ---')

    if DAILY_QUOTA_EXHAUSTED:
        print('🚫 Quota exhausted — stopping.')
        break

    # ── LLM 2: formal specs → draft summary ─────────────────────────
    draft = run_llm2(current_specs)
    save_json(draft, DRAFT_SUMMARY_PATH)

    # ── LLM 3: Judge draft vs ground truth ──────────────────────────
    report = run_llm_judge(ground_truth, draft)
    current_score = report.get('overall_score', 0.0)
    report['iteration'] = iteration
    save_json(report, JUDGE_REPORT_PATH)

    iter_log.append({
        'iteration':      iteration,
        'score':          current_score,
        'passed':         sum(1 for op in report.get('operation_scores', []) if op['status'] == 'PASS'),
        'failed':         sum(1 for op in report.get('operation_scores', []) if op['status'] == 'FAIL'),
        'contradictions': sum(1 for op in report.get('operation_scores', []) if op.get('contradiction_detected')),
        'feedback':       report.get('overall_feedback', ''),
    })

    flag = '✅' if current_score >= SIMILARITY_THRESHOLD else '❌'
    print(f'\n  {flag} Score: {current_score:.4f}  (threshold: {SIMILARITY_THRESHOLD})')

    if current_score >= SIMILARITY_THRESHOLD:
        print('  🎉 THRESHOLD REACHED!')
        break

    if iteration == MAX_ITERATIONS:
        print('  ⚠️  Max iterations reached.')
        break

    # ── Refinement: feed Judge diagnostic feedback back into LLM 1 ──
    print(f'  🔁 Refining for iteration {iteration+1}...')
    extra = build_refinement(report)
    ann   = step1_annotate(all_operations, extra=extra)
    lift  = step2_lift(ann)
    spec  = step3_translate(lift)
    current_specs = spec
    save_json(current_specs, f'/kaggle/working/formal_specifications_iter{iteration+1}.json')
    time.sleep(3)

print('\n' + '='*60 + '\n  DONE\n' + '='*60)
for log in iter_log:
    f = '✅' if log['score'] >= SIMILARITY_THRESHOLD else '❌'
    print(f"  Iter {log['iteration']}: score={log['score']:.4f} {f}  "
          f"passed={log['passed']} failed={log['failed']} contradictions={log['contradictions']}")
save_json(iter_log, ITER_LOG_PATH)
print(f'\n  Final score     : {current_score:.4f}')
print(f'  Result          : {"PASSED ✅" if current_score >= SIMILARITY_THRESHOLD else "NOT REACHED ⚠️"}')


  ITERATIVE PIPELINE  threshold=0.9  max_iter=5

--- ITERATION 1/5 ---

LLM 2 — SUMMARISER
   ✅ 4 sections, 20 ops
   💾 Saved → /kaggle/working/draft_summary.json

BERTSCORE JUDGE
   Ground truth ops : 21
   Draft ops        : 20
   Common ops       : 20
   Missing from draft: ['3-C3']
   Computing STS scores...
   Computing NLI scores...

   BERTSCORE JUDGE RESULTS  |  threshold = 0.9
operationId status  score  sts_score  nli_score     nli_label
        1-A   FAIL 0.8903     0.7833     0.9973    entailment
        1-B   FAIL 0.8565     0.8199     0.8931    entailment
        1-C   FAIL 0.3910     0.7776     0.0045 contradiction
        2-k   FAIL 0.8881     0.7801     0.9962    entailment
        3-A   FAIL 0.8793     0.8381     0.9205    entailment
       3-B1   PASS 0.9125     0.8272     0.9978    entailment
       3-B2   PASS 0.9482     0.8993     0.9971    entailment
       3-B3   PASS 0.9303     0.8638     0.9968    entailment
       3-C1   PASS 0.9540     0.9114     0.9965    e

## 8. Inspect Results

In [ ]:
print('\n📋 SAMPLE FORMAL SPECS (first 3)')
for sp in current_specs[:3]:
    print(f"\n[{sp.get('LABEL','?')}] {sp.get('operationName','?')} ({sp.get('operationId','?')})")
    print(f"  Pre : {sp.get('Precondition','—')}")
    print(f"  Fn  : {sp.get('Function','—')}")
    print(f"  Post: {sp.get('Postcondition','—')}")

In [ ]:
draft = load_json(DRAFT_SUMMARY_PATH)
print('\n📋 DRAFT SUMMARY')
for s in draft.get('sections', []):
    ops = s.get('operations', [])
    print(f"  §{s.get('sectionId')}: {s.get('sectionTitle')} — {len(ops)} ops")
    for op in ops:
        print(f"    [{op.get('operationId')}] {op.get('name')}")

In [ ]:
report = load_json(JUDGE_REPORT_PATH)
print('\n📊 LLM JUDGE REPORT (final iteration)')
print(f"  Score      : {report.get('overall_score')}")
print(f"  Judge type : {report.get('judge_type')} (Multi-Dimensional Formal Methods LLM Judge)")
print(f"  Feedback   : {report.get('overall_feedback', '')}")
print(f"  Missing    : {report.get('missing_operations', [])}")
print('\n  Per-operation:')
for op in report.get('operation_scores', []):
    icon = '✅' if op.get('status') == 'PASS' else ('🔴' if op.get('contradiction_detected') else '⚠️ ')
    print(f"  {icon} [{op.get('operationId')}]  score={op.get('score',0):.4f}  "
          f"pre={op.get('precondition_score',0):.2f}  post={op.get('postcondition_score',0):.2f}  "
          f"status={op.get('status','?')}")
    if op.get('issues'):
        print(f"     Discrepancies: {'; '.join(op.get('issues'))}")
    if op.get('feedback'):
        print(f"     Feedback     : {op.get('feedback')}")

In [ ]:
# Save full per-operation results to CSV
import pandas as pd

report = load_json(JUDGE_REPORT_PATH)
op_scores = report.get('operation_scores', [])

if op_scores:
    df = pd.DataFrame(op_scores)
    csv_path = os.path.join(OUTPUT_DIR, 'judge_results.csv')
    df.to_csv(csv_path, index=False)
    print(f'Full results saved → {csv_path}')
    display_cols = [c for c in ['operationId', 'status', 'score', 'precondition_score', 'postcondition_score', 'state_vars_score', 'purpose_score', 'contradiction_detected'] if c in df.columns]
    print(df[display_cols].to_string(index=False))

# Save failed-ops JSON for manual inspection or retry
failed_ops = [op for op in op_scores if op.get('status') == 'FAIL']
if failed_ops:
    failed_ids = [op['operationId'] for op in failed_ops]
    failed_payload = {
        'similarity_threshold':  SIMILARITY_THRESHOLD,
        'failed_operation_ids':  failed_ids,
        'operations': [
            op
            for section in ground_truth['sections']
            for op in section['operations']
            if op['operationId'] in failed_ids
        ]
    }
    failed_path = os.path.join(OUTPUT_DIR, 'failed_ops.json')
    with open(failed_path, 'w') as f:
        json.dump(failed_payload, f, indent=2)
    print(f'Failed ops payload saved → {failed_path}')
else:
    print('✅ All operations passed. No failed ops to save.')